In [0]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": -23.55,
    "longitude": -46.63,
    "current_weather": True
}

response = requests.get(url, params=params)
data = response.json()

print(data)

In [0]:
from datetime import datetime

cities = {
    "Sao_Paulo": (-23.55, -46.63),
    "Rio_de_Janeiro": (-22.90, -43.20),
    "Brasilia": (-15.79, -47.88),
    "New_York": (40.71, -74.00),
    "London": (51.50, -0.12),
    "Tokyo": (35.68, 139.65),
    "Paris": (48.85, 2.35),
    "Berlin": (52.52, 13.40),
    "Sydney": (-33.86, 151.20),
    "Dubai": (25.20, 55.27)
}

rows = []

for city, coords in cities.items():
    
    params = {
        "latitude": coords[0],
        "longitude": coords[1],
        "current_weather": True
    }
    
    r = requests.get(url, params=params).json()
    
    weather = r["current_weather"]
    
    rows.append({
        "city": city,
        "latitude": coords[0],
        "longitude": coords[1],
        "temperature": weather["temperature"],
        "windspeed": weather["windspeed"],
        "winddirection": weather["winddirection"],
        "weathercode": weather["weathercode"],
        "time": weather["time"],
        "ingestion_timestamp": datetime.now().isoformat()
    })

print(f"Coletados {len(rows)} registros de {len(cities)} cidades")

In [0]:
df = spark.createDataFrame(rows)

display(df)

In [0]:
# Bronze
df.write.format("delta") \
  .mode("append") \
  .option("mergeSchema", "true") \
  .saveAsTable("weather_bronze")

print("Dados salvos na camada BRONZE")

In [0]:
bronze = spark.table("weather_bronze")
display(bronze)

In [0]:
# Silver
from pyspark.sql.functions import col, to_timestamp

bronze = spark.table("weather_bronze")

silver = bronze.select(
    col("city"),
    col("temperature").cast("double"),
    col("windspeed").cast("double"),
    to_timestamp("time").alias("observation_time")
)

silver.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("weather_silver")

print("Silver salva")

In [0]:
# Ouro
from pyspark.sql.functions import avg, max, min

silver = spark.table("weather_silver")

gold = silver.groupBy("city").agg(
    avg("temperature").alias("avg_temperature"),
    max("temperature").alias("max_temperature"),
    min("temperature").alias("min_temperature")
)

gold.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("weather_gold")

print("Gold salva")

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
spark.sql("SELECT * FROM weather_bronze").show()

In [0]:
spark.sql("SELECT * FROM weather_silver").show()


In [0]:
spark.sql("SELECT * FROM weather_gold ORDER BY avg_temperature DESC").show()